# CASCADE-CRC: Coverage Allocation in Two-Stage Conformal Prediction

Every part of the CASCADE-CRC pipeline is in this notebook: splits, shared model fits,
the conformal quantile, calibration, the five methods, and the experiments built on them.
Two small modules sit next to it:

* `synthetic.py` generates the synthetic data (no modelling).
* `metrics.py` scores intervals: group coverage, CovGap, FSC, length, Clopper-Pearson.

The clinical and freMTPL2 inputs are prepared in the separate data repository and read
here as ready files.

**Reproducing the paper.** Run in an environment with `scikit-learn==1.8.0`
(see `requirements.txt`). Gradient-boosted tree fits changed in 1.9, and with them every
interval. With `MODE = "load"` the notebook reads stored results; with `MODE = "run"` it
recomputes them. The last section compares a rerun with the stored results interval by
interval.

## 1. Setup

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier as HGBC
from sklearn.ensemble import HistGradientBoostingRegressor as HGBR
from venn_abers import VennAbers
from joblib import Parallel, delayed

import metrics
import synthetic

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"      # also silences warnings in parallel workers
if sklearn.__version__ != "1.8.0":
    print("WARNING: scikit-learn %s. The paper's results were produced with 1.8.0; "
          "reruns will not match them exactly." % sklearn.__version__)

In [ ]:
# ----- paths and run mode -----
DATA_DIR = os.environ.get("CASCADE_DATA_DIR", "./data")          # ready files from the data repo
RESULTS_DIR = os.environ.get("CASCADE_RESULTS_DIR", "./results")  # where results are read and written
MODE = os.environ.get("CASCADE_MODE", "load")                    # "load" or "run"
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))

# ----- protocol constants used by every experiment -----
ALPHAS = [0.20, 0.10, 0.05]                  # coverage targets 80%, 90%, 95%
N_SPLITS = 30                                # repetitions for the clinical and freMTPL2 data
REG = dict(max_iter=200, max_depth=3)        # point predictor, difficulty, scale and CQR models
CLF = dict(max_iter=200, max_depth=4)        # Stage 1 classifier
BETA_GRID = np.arange(0.0, 1.05, 0.1)        # CASCADE's beta candidates
G = 3                                        # difficulty groups (tertiles)

DATA_FILES = {"PPMI": "ppmi_ready.csv", "XYZ": "xyz_health_ready.csv", "freMTPL2": "fremtpl2_ready.csv"}
os.makedirs(RESULTS_DIR, exist_ok=True)
print("data:", DATA_DIR, "| results:", RESULTS_DIR, "| mode:", MODE, "| jobs:", N_JOBS)

### Stored results from the submission

The submission's result files use older names and labels. If they are in
`results/submission/`, this cell copies each one once into the name and labels this
notebook uses, so `MODE = "load"` works straight away:

| stored file | becomes |
|---|---|
| `fine_clinical` (a superset of `lowW_clinical` and `alpha_ext_clinical`, identical on shared arms) | `clinical_strength` |
| `factorial_clinical` | `clinical_factorial` |
| `fre_lowW` (a superset of `fre_all_alpha`) | `fre_strength` |
| `factorial_fre` | `fre_factorial` |
| `synth_v2` | `synth_strength` |
| `synth_factorial` | `synth_factorial` |
| `selector_results` | `selection_clinical` |
| `selector_synthetic` | `selection_synthetic` |

The submission stored no freMTPL2 selection results; `selection_fre` is computed here.

The cohort label `UF` becomes `XYZ`. The synthetic oracle rows are relabelled
`(as submitted)`: they carry the oracle bug described in section 8.1.

In [ ]:
LEGACY_DIR = os.environ.get("CASCADE_LEGACY_DIR", os.path.join(RESULTS_DIR, "submission"))


def legacy_label(arm):
    """Map a submission-era arm label to this notebook's label."""
    fixed = {"CONSTANT": "CASCADE-CRC CONSTANT", "STAGE2": "CASCADE-CRC STAGE2",
             "CASCADE": "CASCADE (dev-beta)", "CASCADE dev-only beta": "CASCADE (dev-beta)",
             "Mondrian": "Mondrian (standard)", "MEAN w=0.00": "CASCADE-CRC CONSTANT",
             "MEAN w=1.00": "CASCADE-CRC STAGE2"}
    if arm in fixed:
        return fixed[arm]
    if arm.startswith("SHRINK-") or arm.startswith("Q-X"):
        return "CASCADE-CRC " + arm
    if arm.startswith("MEAN w="):
        return "CASCADE-CRC SHRINK-" + arm[len("MEAN w="):]
    return arm


SUBMISSION_FILES = {"synth_strength": ("synth_v2", "synthetic"),
                    "synth_factorial": ("synth_factorial", "synthetic"),
                    "clinical_strength": ("fine_clinical", "clinical"),
                    "clinical_factorial": ("factorial_clinical", "clinical"),
                    "fre_strength": ("fre_lowW", "fre"),
                    "fre_factorial": ("factorial_fre", "fre"),
                    "selection_clinical": ("selector_results", "selection"),
                    "selection_synthetic": ("selector_synthetic", "selection")}


def import_submission_results():
    for new, (old, kind) in SUBMISSION_FILES.items():
        src, dst = os.path.join(LEGACY_DIR, old + ".csv.gz"), os.path.join(RESULTS_DIR, new + ".csv.gz")
        if os.path.exists(dst) or not os.path.exists(src):
            continue
        d = pd.read_csv(src, low_memory=False)
        if "arm" in d:
            d["arm"] = d.arm.map(legacy_label)
        if kind == "clinical":
            d["cohort"] = d.cohort.replace({"UF": "XYZ"})
        if kind == "selection":
            d["dataset"] = d.dataset.replace({"UF": "XYZ"})
        if kind == "synthetic":
            d["budget"] = d.budget.replace({"UF-sized-sensitivity": "XYZ-sized-sensitivity"})
            d["arm"] = d.arm.replace({"ORACLE-noise": "ORACLE-noise (as submitted)",
                                      "ORACLE-residual-q": "ORACLE-residual-q (as submitted)"})
        d.to_csv(dst, index=False)
        print("imported %s -> %s (%d rows)" % (old, new, len(d)))


import_submission_results()

## 2. Conformal core

Every method below ends in the same step: compute nonconformity scores on the calibration
units, take the split-conformal order statistic, and turn it into an interval.

For a normalized score $|y - \hat f(x)| / s(x)$ the interval is
$\hat f(x) \pm \hat q \, s(x)$, where $\hat q$ is the
$\lceil (n+1)(1-\alpha) \rceil$-th smallest calibration score (infinite when that rank
exceeds $n$).

**CASCADE-CRC calibration.** CASCADE-CRC calibrates each strength with conformal risk
control on the miscoverage loss. Calibration and evaluation use one encounter per patient,
so the loss is binary per unit, and CRC's threshold reduces exactly to the order statistic
above. That is how it is computed here: `scaled_interval` *is* the CRC step for this
protocol.

In [ ]:
def conformal_quantile(scores, alpha):
    """Split-conformal order statistic: the ceil((n+1)(1-alpha))-th smallest score, or +inf."""
    n = len(scores)
    if n == 0:
        return np.inf
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else float(np.sort(scores)[k - 1])


def scaled_interval(res_cal, s_cal, fhat_test, s_test, alpha):
    """Calibrate a scale s(x) on calibration residuals; return (lo, hi) on the test units."""
    q = conformal_quantile(res_cal / s_cal, alpha)
    return fhat_test - q * s_test, fhat_test + q * s_test

## 3. Splits

Every repetition assigns units to four roles:

| role | used for |
|---|---|
| `A` (fit) | point predictor $\hat f$ (on $Z = 1$ units) and the Stage 1 classifier |
| `B` (development hold-out, $D_{hold}$) | Venn-Abers calibration, difficulty and scale models, CQR quantiles, CASCADE's $\beta$, group cut points |
| `cal` | conformal calibration ($Z = 1$ units only) |
| `test` | evaluation ($Z = 1$ units only) |

**Clinical cohorts.** Patients are split 50/30/20 into development, calibration and
evaluation, and development patients 60/40 into `A` and `B`. Calibration and evaluation
draw one encounter per patient. `B`'s Venn-Abers features are cross-fitted across two
patient halves so the scale models never see in-sample Venn-Abers outputs.

**freMTPL2.** The same proportions over policies, which are the units.

**Synthetic.** `synthetic.replication` draws `A`, `B`, `cal` and `test` as independent
samples.

All seeds are fixed per repetition and written out below.

In [ ]:
def clinical_split(pid, z, s):
    """Roles for split s of a clinical cohort, or None if a role is too small."""
    pats = np.unique(pid)
    pp = np.random.default_rng(3000 + s).permutation(pats)
    n = len(pp)
    dev, cal, te = set(pp[:int(.5 * n)]), set(pp[int(.5 * n):int(.8 * n)]), set(pp[int(.8 * n):])
    M = {k: np.array([p in v for p in pid]) for k, v in [("dev", dev), ("cal", cal), ("te", te)]}
    di = np.flatnonzero(M["dev"])
    dp = np.unique(pid[di])
    np.random.default_rng(4000 + s).shuffle(dp)
    fitp = set(dp[:int(.6 * len(dp))])
    mf = np.array([p in fitp for p in pid[di]])
    A, B = di[mf], di[~mf]
    if (z[A] == 1).sum() < 30 or (z[B] == 1).sum() < 40:
        return None
    # patient halves of B for cross-fitted Venn-Abers features
    hp = np.unique(pid[B])
    np.random.default_rng(5000 + s).shuffle(hp)
    half = set(hp[:len(hp) // 2])
    va_half = np.array([p in half for p in pid[B]])
    # one encounter per patient among eligible (Z = 1) encounters, calibration then evaluation
    rs = np.random.default_rng(9500 + s)
    def one_per_patient(mask):
        ix = np.flatnonzero(mask)
        ix = ix[z[ix] == 1]
        return np.array([rs.choice(ix[pid[ix] == p]) for p in np.unique(pid[ix])])
    ci, ti = one_per_patient(M["cal"]), one_per_patient(M["te"])
    if len(ci) < 30 or len(ti) < 15:
        return None
    return dict(A=A, B=B, cal=ci, test=ti, va_half=va_half)


def policy_split(z, s):
    """Roles for split s of freMTPL2 (units are policies), or None if a role is too small."""
    n = len(z)
    pm = np.random.default_rng(s).permutation(n)
    dev, cal, te = pm[:int(.5 * n)], pm[int(.5 * n):int(.8 * n)], pm[int(.8 * n):]
    dd = dev.copy()
    np.random.default_rng(1000 + s).shuffle(dd)
    A, B = dd[:int(.6 * len(dd))], dd[int(.6 * len(dd)):]
    c1, t1 = cal[z[cal] == 1], te[z[te] == 1]
    if min((z[A] == 1).sum(), (z[B] == 1).sum(), len(c1), len(t1)) < 40:
        return None
    return dict(A=A, B=B, cal=c1, test=t1, va_half=None)


def synthetic_roles(rep_data):
    """Roles for one synthetic replication, or None if a role is too small."""
    z, R = rep_data["z"], rep_data["roles"]
    c1, t1 = R["C"][z[R["C"]] == 1], R["T"][z[R["T"]] == 1]
    if (z[R["A"]] == 1).sum() < 30 or (z[R["B"]] == 1).sum() < 30 or len(c1) < 25:
        return None
    return dict(A=R["A"], B=R["B"], cal=c1, test=t1, va_half=None)

## 4. Shared fits

Fitted once per repetition and shared by every method and coverage target:

* $\hat f$: point predictor on `A`'s $Z = 1$ units.
* Stage 1 classifier on all of `A`, then Venn-Abers on `B`. Each unit gets the raw
  probability, the Venn-Abers probability $(p_0 + p_1)/2$, the Venn-Abers width
  $u = p_1 - p_0$, and $p_0$, $p_1$.
* $\tilde\sigma$: the mean-residual difficulty model, fitted to $|y - \hat f(x)|$ on `B`,
  floored at its 5th percentile on `B` and normalized by its mean there.
* The grouping model: an independent difficulty model (seed + 1) whose tertile cut points
  on `B` define the evaluation groups, applied unchanged to every method.

In [ ]:
def fit_shared(X, y, z, roles, seed):
    A, B, ci, ti, va_half = roles["A"], roles["B"], roles["cal"], roles["test"], roles["va_half"]
    a1, b1 = A[z[A] == 1], B[z[B] == 1]
    fh = HGBR(**REG, random_state=seed).fit(X[a1], y[a1])
    clf = HGBC(**CLF, random_state=seed).fit(X[A], z[A])

    # Stage 1 features: raw probability, VA probability, VA width, p0, p1
    F = np.full((len(X), 5), np.nan)
    F[:, 0] = clf.predict_proba(X)[:, 1]
    def venn_abers(ix):
        p = clf.predict_proba(X[ix])[:, 1]
        V = VennAbers()
        V.fit(np.column_stack([1 - p, p]), z[ix])
        return V
    def set_va(ix, V):
        p = clf.predict_proba(X[ix])[:, 1]
        _, q = V.predict_proba(np.column_stack([1 - p, p]))
        F[ix, 1] = (q[:, 0] + q[:, 1]) / 2
        F[ix, 2] = q[:, 1] - q[:, 0]
        F[ix, 3], F[ix, 4] = q[:, 0], q[:, 1]
    V_all = venn_abers(B)
    if va_half is None:
        set_va(B, V_all)
    else:                                    # cross-fit B's features across patient halves
        set_va(B[va_half], venn_abers(B[~va_half]))
        set_va(B[~va_half], venn_abers(B[va_half]))
    set_va(ci, V_all)
    set_va(ti, V_all)

    # difficulty: mean-residual model on B (sigma-tilde), with floor and mean-one normalizer
    res_b = np.abs(y[b1] - fh.predict(X[b1]))
    mm = HGBR(**REG, random_state=seed).fit(X[b1], res_b)
    floor = max(float(np.percentile(np.maximum(mm.predict(X[b1]), 0), 5)), 1e-6)
    norm = float(np.maximum(mm.predict(X[b1]), floor).mean())

    # grouping model: independent difficulty model, tertile cut points on B
    gm = HGBR(**REG, random_state=seed + 1).fit(X[b1], res_b)
    g_hold = gm.predict(X[b1])
    cuts = np.percentile(g_hold, [33.3, 66.7])
    g_test_score = gm.predict(X[ti])

    return dict(
        X=X, y=y, seed=seed, a1=a1, b1=b1, ci=ci, ti=ti, F=F, res_b=res_b,
        fh=fh, fhat_t=fh.predict(X[ti]), res_c=np.abs(y[ci] - fh.predict(X[ci])),
        sig_c=np.maximum(mm.predict(X[ci]), floor), sig_t=np.maximum(mm.predict(X[ti]), floor),
        norm=norm, u_mean=float(np.mean(F[b1, 2])),
        va_cuts=np.percentile(F[b1, 2], [33.3, 66.7]),
        g_cal=np.clip(np.digitize(gm.predict(X[ci]), cuts), 0, G - 1),
        g_test=np.clip(np.digitize(g_test_score, cuts), 0, G - 1),
        g_test_score=g_test_score, g_hold=g_hold)

## 5. Methods

Each method takes the shared fits and a coverage target and returns `(lo, hi)` on the
evaluation units.

* **Split conformal**: constant width.
* **Mondrian**: split conformal within each difficulty group.
* **CQR**: conformalized quantile regression, quantiles fitted on `B`.
* **CASCADE**: scale $\mathrm{clip}(1 + \beta(u/\bar u - 1), 0.2, 5)$ in the Venn-Abers
  width $u$, with $\beta$ chosen on two halves of `B` (coverage closest to target, ties to
  shorter).
* **CASCADE-CRC($w$)**: scale $s_w(x) = (1 - w) + w\,\tilde\sigma(x)/\bar{\tilde\sigma}$.
  $w = 0$ is constant width (`CONSTANT`), $w = 1$ is fully scaled (`STAGE2`).

Two families swap the difficulty score $\tilde\sigma$ for a quantile model of the
residual on a set of inputs (the *signals* below):

* raw (`CASCADE-CRC <signal>`): the quantile model's prediction used directly as the scale;
* strength-graded (`SIG <signal> w=...`): floored and normalized like $\tilde\sigma$, then
  plugged into $s_w$.

In [ ]:
SIGNALS = {
    "Q-X":               lambda c, i: c["X"][i],
    "Q-X+RAW-PROB":      lambda c, i: np.column_stack([c["X"][i], c["F"][i, 0]]),
    "Q-X+VA-PROB":       lambda c, i: np.column_stack([c["X"][i], c["F"][i, 1]]),
    "Q-X+VA-PROB+WIDTH": lambda c, i: np.column_stack([c["X"][i], c["F"][i, 1], c["F"][i, 2]]),
    "Q-X+VA-P1":         lambda c, i: np.column_stack([c["X"][i], c["F"][i, 4]]),
    "Q-X+VA-P0P1":       lambda c, i: np.column_stack([c["X"][i], c["F"][i, 3], c["F"][i, 4]]),
}


def split_conformal(c, alpha):
    q = conformal_quantile(c["res_c"], alpha)
    return c["fhat_t"] - q, c["fhat_t"] + q


def mondrian(c, alpha):
    h = np.zeros(len(c["ti"]))
    for j in range(G):
        h[c["g_test"] == j] = conformal_quantile(c["res_c"][c["g_cal"] == j], alpha)
    return c["fhat_t"] - h, c["fhat_t"] + h


def cqr(c, alpha):
    X, y, b1, s = c["X"], c["y"], c["b1"], c["seed"]
    qlo = HGBR(**REG, loss="quantile", quantile=alpha / 2, random_state=s).fit(X[b1], y[b1])
    qhi = HGBR(**REG, loss="quantile", quantile=1 - alpha / 2, random_state=s).fit(X[b1], y[b1])
    ci, ti = c["ci"], c["ti"]
    lo_c, hi_c = qlo.predict(X[ci]), qhi.predict(X[ci])
    q = conformal_quantile(np.maximum(lo_c - y[ci], y[ci] - hi_c), alpha)
    return qlo.predict(X[ti]) - q, qhi.predict(X[ti]) + q


def cascade(c, alpha):
    """CASCADE with beta chosen on two halves of B. Returns (lo, hi, beta)."""
    X, y, fh, u, ub = c["X"], c["y"], c["fh"], c["F"][:, 2], c["u_mean"]
    hh = c["b1"].copy()
    np.random.default_rng(7000 + c["seed"]).shuffle(hh)
    D1, D2 = hh[:len(hh) // 2], hh[len(hh) // 2:]
    r1, r2 = np.abs(y[D1] - fh.predict(X[D1])), np.abs(y[D2] - fh.predict(X[D2]))
    scale = lambda uu, bt: np.clip(1 + bt * (uu / ub - 1), 0.2, 5.0)
    best, err, length = 0.0, np.inf, np.inf
    for bt in BETA_GRID:
        hw = conformal_quantile(r1 / scale(u[D1], bt), alpha) * scale(u[D2], bt)
        cv, L = float(np.mean(r2 <= hw)), float(np.mean(2 * hw))
        if cv < 1 - alpha or not np.isfinite(L):
            continue
        e = abs(cv - (1 - alpha))
        if e < err - 1e-4 or (abs(e - err) <= 1e-4 and L < length):
            best, err, length = bt, e, L
    lo, hi = scaled_interval(c["res_c"], scale(u[c["ci"]], best), c["fhat_t"],
                             scale(u[c["ti"]], best), alpha)
    return lo, hi, best


def strength_scale(sigma, w, norm):
    """CASCADE-CRC scale s_w = (1 - w) + w * sigma / norm."""
    return (1 - w) + w * sigma / norm


def signal_quantile_model(c, name, alpha):
    f = SIGNALS[name]
    return HGBR(**REG, loss="quantile", quantile=1 - alpha,
                random_state=c["seed"]).fit(f(c, c["b1"]), c["res_b"]), f

## 6. Runner

An experiment is a list of arms. `run_methods` builds them for one repetition and one
coverage target; `run_repetition` fits the shared components once and loops over targets.

In [ ]:
def run_methods(c, alpha, arms):
    """Intervals for every requested arm. Returns ({label: (lo, hi)}, {"beta": ...})."""
    out, info = {}, {}
    ci, ti, fhat, res = c["ci"], c["ti"], c["fhat_t"], c["res_c"]
    if "Split" in arms.get("baselines", ()):
        out["Split"] = split_conformal(c, alpha)
    if "Mondrian" in arms.get("baselines", ()):
        out["Mondrian (standard)"] = mondrian(c, alpha)
    if "CQR" in arms.get("baselines", ()):
        out["CQR"] = cqr(c, alpha)
    if "CASCADE" in arms.get("baselines", ()):
        lo, hi, info["beta"] = cascade(c, alpha)
        out["CASCADE (dev-beta)"] = (lo, hi)
    # CASCADE-CRC strength family on the mean-residual difficulty score
    out["CASCADE-CRC CONSTANT"] = scaled_interval(res, np.ones(len(ci)), fhat, np.ones(len(ti)), alpha)
    out["CASCADE-CRC STAGE2"] = scaled_interval(res, c["sig_c"], fhat, c["sig_t"], alpha)
    for w in arms.get("strengths", ()):
        out["CASCADE-CRC SHRINK-%.2f" % w] = scaled_interval(
            res, strength_scale(c["sig_c"], w, c["norm"]), fhat,
            strength_scale(c["sig_t"], w, c["norm"]), alpha)
    # quantile-model signals used directly as the scale
    for name in arms.get("signals_raw", ()):
        m, f = signal_quantile_model(c, name, alpha)
        out["CASCADE-CRC " + name] = scaled_interval(
            res, np.maximum(m.predict(f(c, ci)), 1e-6), fhat,
            np.maximum(m.predict(f(c, ti)), 1e-6), alpha)
    # quantile-model signals, floored and normalized on B, then graded by strength
    for name in arms.get("signals_graded", ()):
        m, f = signal_quantile_model(c, name, alpha)
        gb = m.predict(f(c, c["b1"]))
        fl = max(float(np.percentile(np.maximum(gb, 0), 5)), 1e-6)
        nk = float(np.maximum(gb, fl).mean())
        s_c, s_t = np.maximum(m.predict(f(c, ci)), fl) / nk, np.maximum(m.predict(f(c, ti)), fl) / nk
        for w in arms["graded_strengths"]:
            out["SIG %s w=%.2f" % (name, w)] = scaled_interval(
                res, strength_scale(s_c, w, 1.0), fhat, strength_scale(s_t, w, 1.0), alpha)
    return out, info


def run_repetition(X, y, z, roles, seed, alphas, arms, extra=None):
    """Shared fits once, then every arm at every target.

    `extra(c, alpha)` may add experiment-specific arms (the synthetic oracles).
    Returns the shared fits and a list of (alpha, intervals, info).
    """
    c = fit_shared(X, y, z, roles, seed)
    results = []
    for alpha in alphas:
        intervals, info = run_methods(c, alpha, arms)
        if extra is not None:
            intervals.update(extra(c, alpha))
        results.append((alpha, intervals, info))
    return c, results


def run_or_load(name, compute, needs=()):
    """Load results/<name>.csv.gz in "load" mode; otherwise compute, save and return.

    Returns None, with a message, when the stored file or a required dataset is missing.
    """
    path = os.path.join(RESULTS_DIR, name + ".csv.gz")
    if MODE == "load":
        if not os.path.exists(path):
            print("%s: no stored results at %s (set MODE = 'run' to compute)" % (name, path))
            return None
        return pd.read_csv(path, low_memory=False)
    missing = [d for d in needs if not os.path.exists(os.path.join(DATA_DIR, DATA_FILES[d]))]
    if missing:
        print("%s: skipped, missing data for %s" % (name, ", ".join(missing)))
        return None
    t0 = time.time()
    df = compute()
    df.to_csv(path, index=False)
    print("%s: %d rows in %.0fs -> %s" % (name, len(df), time.time() - t0, path))
    return df

## 7. Data

The data repository writes one ready file per dataset with these columns:

| column | meaning |
|---|---|
| `unit_id` | patient (clinical) or policy (freMTPL2) |
| `y` | outcome: log dose ratio (PPMI), normalized percent change (XYZ Health), log1p claim amount (freMTPL2) |
| `prev` | previous dose (clinical; `0` marks a transition from a recorded zero) |
| `amount` | claim amount in euros (freMTPL2) |
| every other column | a feature, already encoded and scaled, in model order |

The routing indicator is $Z = 1\{y \neq 0\}$. Rows must stay in the order the data
repository writes them: the splits index rows by position. Files are read with
`float_precision="round_trip"`; pandas' default parser can change the last bit of a float,
which is enough to change a fitted tree.

In [ ]:
RESERVED = {"unit_id", "y", "prev", "amount"}


def load_ready(name):
    # round_trip parsing returns exactly the floats the data repository wrote
    d = pd.read_csv(os.path.join(DATA_DIR, DATA_FILES[name]), low_memory=False, float_precision="round_trip")
    feats = [col for col in d.columns if col not in RESERVED]
    y = d["y"].values.astype(float)
    return dict(X=d[feats].values.astype(float), y=y, z=(y != 0).astype(int),
                unit_id=d["unit_id"].values,
                prev=d["prev"].values if "prev" in d else None,
                amount=d["amount"].values if "amount" in d else None, features=feats)


for name, f in DATA_FILES.items():
    print("%-9s %s" % (name, "found" if os.path.exists(os.path.join(DATA_DIR, f)) else "missing: " + f))

## 8. Experiments

| experiment | paper | arms |
|---|---|---|
| `clinical_strength` | Table 1, Tables 6-9, robustness | baselines, $w \in \{0.05, \dots, 0.95\}$, raw signals |
| `clinical_factorial` | Tables 3, 10 | baselines, $w \in \{0.25, 0.5, 0.75\}$, strength-graded signals |
| `fre_strength` | Table 2 | baselines, $w \in \{0.05, \dots, 0.75\}$, raw signals |
| `fre_factorial` | Tables 3, 10 | as `clinical_factorial` |
| `synth_strength` | Table 2 | baselines, $w \in \{0.15, \dots, 0.75\}$, raw signals, oracles |
| `synth_factorial` | Tables 3, 10 | Split, Mondrian, CQR, $w$ grid, strength-graded signals, oracle |

The appendix C strength-selection experiment and the figure scripts are not yet in this
notebook.

In [ ]:
BASELINES = ["Split", "Mondrian", "CQR", "CASCADE"]
RAW_SIGNALS = ["Q-X", "Q-X+RAW-PROB", "Q-X+VA-PROB", "Q-X+VA-PROB+WIDTH"]
ALL_SIGNALS = RAW_SIGNALS + ["Q-X+VA-P1", "Q-X+VA-P0P1"]
GRADED_W = [0.0, 0.15, 0.25, 0.50, 0.75, 1.0]

ARMS = {
    "clinical_strength": dict(baselines=BASELINES, strengths=[round(0.05 * i, 2) for i in range(1, 20)],
                              signals_raw=RAW_SIGNALS),
    "clinical_factorial": dict(baselines=BASELINES, strengths=[0.25, 0.50, 0.75],
                               signals_graded=ALL_SIGNALS, graded_strengths=GRADED_W),
    "fre_strength": dict(baselines=BASELINES, strengths=[0.05, 0.10, 0.15, 0.25, 0.50, 0.75],
                         signals_raw=RAW_SIGNALS),
    "fre_factorial": dict(baselines=BASELINES, strengths=[0.25, 0.50, 0.75],
                          signals_graded=ALL_SIGNALS, graded_strengths=GRADED_W),
    "synth_strength": dict(baselines=BASELINES, strengths=[0.15, 0.25, 0.50, 0.75],
                           signals_raw=RAW_SIGNALS),
    "synth_factorial": dict(baselines=["Split", "Mondrian", "CQR"], strengths=[0.15, 0.25, 0.50, 0.75],
                            signals_graded=ALL_SIGNALS, graded_strengths=GRADED_W),
}

### 8.1 Synthetic data

2 calibration budgets (92 and 361 eligible units, sized like XYZ Health and PPMI)
x 2 heteroscedasticity levels x 3 Stage 1 relevance levels x signed/nonnegative outcomes,
40 replications each, at three targets. Groups use the *true* noise scale (`_true`
columns) and the estimated difficulty (`_pred` columns).

Two oracle arms, available only here:

* `ORACLE-noise`: CASCADE-CRC scaled by the true noise scale $\sigma(x)$.
* `ORACLE-residual-q`: the empirical $(1-\alpha)$ quantile of calibration residuals within
  each true-difficulty tertile (not conformal; a reference point).

**Oracle bug in the submitted runs.** The submission's code calibrated both oracles with
the *evaluation* sample's noise scales read at the calibration units' positions, so the
calibration scales belonged to unrelated units. The arms below use each calibration unit's
own $\sigma(x)$. Set `ORACLE_AS_SUBMITTED = True` to reproduce the stored numbers; no other
arm is affected.

In [ ]:
SYNTH_SETTINGS = dict(
    budgets=[("XYZ-sized-sensitivity", 92), ("PPMI-sized-sensitivity", 361)],
    hets=[("homogeneous", 0.0), ("heterogeneous", 0.6)],
    relevs=[("informative", 0.8), ("irrelevant", 0.0), ("misleading", -0.8)],
    signs=[("signed", True), ("nonnegative", False)])


ORACLE_AS_SUBMITTED = False


def synthetic_oracles(which):
    def extra(c, alpha):
        sig, ci, ti, out = c["sigma"], c["ci"], c["ti"], {}
        sig_c = c["sigma_cal_submitted"] if ORACLE_AS_SUBMITTED else sig[ci]
        if "ORACLE-noise" in which:
            out["ORACLE-noise"] = scaled_interval(c["res_c"], np.maximum(sig_c, 1e-6), c["fhat_t"],
                                                  np.maximum(sig[ti], 1e-6), alpha)
        if "ORACLE-residual-q" in which:
            g_c = np.clip(np.digitize(sig_c, c["true_cuts"]), 0, G - 1)
            q = np.array([np.quantile(c["res_c"][g_c == j], 1 - alpha) if (g_c == j).sum() > 5
                          else np.quantile(c["res_c"], 1 - alpha) for j in range(G)])
            h = q[c["g_true"]]
            out["ORACLE-residual-q"] = (c["fhat_t"] - h, c["fhat_t"] + h)
        return out
    return extra


def synthetic_replication(arms_name, oracles, N, het, relev, signed, rep, labels, alphas=ALPHAS):
    """One replication at every target, scored per arm (one row per arm per target)."""
    D = synthetic.replication(N, het, relev, signed, rep)
    roles = synthetic_roles(D)
    if roles is None:
        return []
    sig, R = D["sigma"], D["roles"]
    true_cuts = np.percentile(sig[roles["test"]], [33.3, 66.7])
    # the submission's oracle read the evaluation sample's sigma at the calibration positions
    sig_cal_submitted = sig[R["T"][0] + (roles["cal"] - R["C"][0])]
    def extra(c, alpha):                       # oracle arms need the true noise scale
        c.update(sigma=sig, true_cuts=true_cuts, sigma_cal_submitted=sig_cal_submitted,
                 g_true=np.clip(np.digitize(sig[c["ti"]], true_cuts), 0, G - 1))
        return synthetic_oracles(oracles)(c, alpha)
    c, results = run_repetition(D["X"], D["y"], D["z"], roles, rep, alphas, ARMS[arms_name], extra)
    yt = D["y"][c["ti"]]
    base = dict(N=N, het=het, relev=relev, signed=signed, rep=rep, n_gen_cal=N,
                n_eligible_cal=len(c["ci"]), n_eval=len(c["ti"]), n_dev_elig=len(c["b1"]))
    rows = []
    for alpha, intervals, info in results:
        for arm, (lo, hi) in intervals.items():
            cov = (yt >= lo) & (yt <= hi)
            L = metrics.interval_length(lo, hi)
            pt = metrics.group_coverage(cov, c["g_true"], G)
            pp = metrics.group_coverage(cov, c["g_test"], G)
            rows.append(dict(base, arm=arm, coverage=float(cov.mean()), length=float(np.nanmean(L)),
                             covgap_true=metrics.covgap(pt, alpha), fsc_true=metrics.fsc(pt),
                             covgap_pred=metrics.covgap(pp, alpha), fsc_pred=metrics.fsc(pp),
                             empty=float((hi < lo).mean()), inf=float((~np.isfinite(L)).mean()),
                             **{"ct%d" % j: pt[j] for j in range(G)}, **labels, alpha=alpha))
    return rows


def synthetic_experiment(arms_name, oracles, settings, reps):
    tasks = [(N, h, r, sg, rep, dict(budget=bn, het_lab=hn, relev_lab=rn, signed_lab=sn))
             for (bn, N) in settings["budgets"] for (hn, h) in settings["hets"]
             for (rn, r) in settings["relevs"] for (sn, sg) in settings["signs"] for rep in range(reps)]
    out = Parallel(n_jobs=N_JOBS)(delayed(synthetic_replication)(arms_name, oracles, *t) for t in tasks)
    return pd.DataFrame([row for rows in out for row in rows])


synth_strength = run_or_load("synth_strength", lambda: synthetic_experiment(
    "synth_strength", ["ORACLE-noise", "ORACLE-residual-q"], SYNTH_SETTINGS, reps=40))

synth_factorial = run_or_load("synth_factorial", lambda: synthetic_experiment(
    "synth_factorial", ["ORACLE-residual-q"],
    dict(SYNTH_SETTINGS, budgets=[("PPMI-sized-sensitivity", 361)], hets=[("heterogeneous", 0.6)]),
    reps=25))

### 8.2 Clinical cohorts

PPMI and XYZ Health, 30 patient-level splits, three targets. One row per evaluated
encounter per split per arm, with the evaluation groups stored alongside:

* `g_d2`: difficulty tertile (the paper's groups); `g_d5`, `g_d7`: quintiles and septiles
  from the same score; `d2`: the score itself.
* `g_va`: Venn-Abers width tertile; `g_pz`: transition from a recorded zero.

In [ ]:
def clinical_repetition(D, cohort, arms_name, s, alphas=ALPHAS):
    roles = clinical_split(D["unit_id"], D["z"], s)
    if roles is None:
        return []
    c, results = run_repetition(D["X"], D["y"], D["z"], roles, s, alphas, ARMS[arms_name])
    ti = c["ti"]
    yt, prev = D["y"][ti], D["prev"][ti]
    d2h, d2t = c["g_hold"], c["g_test_score"]
    groups = dict(
        g_d2=c["g_test"],
        g_d5=np.clip(np.digitize(d2t, np.percentile(d2h, [20, 40, 60, 80])), 0, 4),
        g_d7=np.clip(np.digitize(d2t, np.percentile(d2h, [100 / 7 * i for i in range(1, 7)])), 0, 6),
        d2=d2t,
        g_va=np.clip(np.digitize(c["F"][ti, 2], c["va_cuts"]), 0, G - 1),
        g_pz=(prev == 0).astype(int))
    frames = []
    for alpha, intervals, info in results:
        for arm, (lo, hi) in intervals.items():
            L = hi - lo
            frames.append(pd.DataFrame(dict(
                cohort=cohort, alpha=alpha, split=s, arm=arm, patient=D["unit_id"][ti], lo=lo, hi=hi,
                covered=(yt >= lo) & (yt <= hi), set_length=np.maximum(L, 0.0), signed_diff=L,
                empty=hi < lo, infinite=~np.isfinite(np.maximum(L, 0.0)), **groups,
                prev=prev, y=yt, fhat=c["fhat_t"])))
    return frames


def clinical_experiment(arms_name, cohorts=("PPMI", "XYZ"), splits=N_SPLITS):
    data = {coh: load_ready(coh) for coh in cohorts}
    tasks = [(coh, s) for coh in cohorts for s in range(splits)]
    out = Parallel(n_jobs=N_JOBS)(delayed(clinical_repetition)(data[coh], coh, arms_name, s)
                                  for coh, s in tasks)
    return pd.concat([f for frames in out for f in frames], ignore_index=True)


clinical_strength = run_or_load("clinical_strength", lambda: clinical_experiment("clinical_strength"),
                                needs=["PPMI", "XYZ"])
clinical_factorial = run_or_load("clinical_factorial", lambda: clinical_experiment("clinical_factorial"),
                                 needs=["PPMI", "XYZ"])

### 8.3 freMTPL2

150,000 policies (sampled once in the data repository), 30 policy-level splits. One row per
split per arm. Groups are the difficulty tertiles; `money_len` is the median interval
length in euros.

In [ ]:
def fre_repetition(D, arms_name, s, alphas=ALPHAS):
    roles = policy_split(D["z"], s)
    if roles is None:
        return []
    c, results = run_repetition(D["X"], D["y"], D["z"], roles, s, alphas, ARMS[arms_name])
    yt, gt = D["y"][c["ti"]], c["g_test"]
    rows = []
    for alpha, intervals, info in results:
        for arm, (lo, hi) in intervals.items():
            cov = (yt >= lo) & (yt <= hi)
            L = metrics.interval_length(lo, hi)
            per = metrics.group_coverage(cov, gt, G)
            lens = metrics.group_mean(L, gt, G)
            rows.append(dict(alpha=alpha, split=s, arm=arm, n_cal=len(c["ci"]), n_te=len(c["ti"]),
                             n_dev_pos=len(c["b1"]), coverage=float(cov.mean()), length=float(np.mean(L)),
                             money_len=float(np.median(np.expm1(hi) - np.maximum(np.expm1(lo), 0.0))),
                             beta=info["beta"] if arm == "CASCADE (dev-beta)" else np.nan,
                             covgap=metrics.covgap(per, alpha), fsc=metrics.fsc(per),
                             empty=float((hi < lo).mean()), inf=float((~np.isfinite(L)).mean()),
                             **{"c%d" % j: per[j] for j in range(G)}, **{"l%d" % j: lens[j] for j in range(G)}))
    return rows


def fre_experiment(arms_name, splits=N_SPLITS):
    D = load_ready("freMTPL2")
    out = Parallel(n_jobs=N_JOBS)(delayed(fre_repetition)(D, arms_name, s) for s in range(splits))
    return pd.DataFrame([row for rows in out for row in rows])


fre_strength = run_or_load("fre_strength", lambda: fre_experiment("fre_strength"), needs=["freMTPL2"])
fre_factorial = run_or_load("fre_factorial", lambda: fre_experiment("fre_factorial"), needs=["freMTPL2"])

### 8.4 Strength selection (Appendix C)

Can the strength be chosen without evaluation outcomes? Each split's calibration units
are halved by a fixed, outcome-independent rule (seed 31000 + s) *before* eligibility
filtering: one half calibrates the candidates $w \in \{0, 0.25, 0.5, 0.75, 1\}$, the other
(the selection half) is used to choose among them. The evaluation set is never used.
Groups here are tertiles of the mean-residual difficulty model itself.

Rules:

* `EMPIRICAL`: the candidate with the smallest observed CovGap on the selection half.
* `FIXED-CONFIDENCE` (the certifying rule of Appendix C): simultaneous Clopper-Pearson
  intervals for every candidate's group coverages at total error $\delta/15$,
  $\delta = 0.05$. It certifies an adaptive candidate only when its largest plausible
  CovGap $U(w)$ is below constant width's smallest plausible one $L(0)$, and then takes the
  smallest $U(w)$; otherwise it keeps $w = 0$.
* `E-PROCESS-FULL` and `E-PROCESS-EARLY`: the same certificate with time-uniform
  (anytime-valid) confidence sets in place of Clopper-Pearson, evaluated after all
  selection units or at the first unit where it fires.
* `FIXED w=...`: every candidate, for reference.

Ties go to the smaller $w$.

In [ ]:
SEL_W = [0.0, 0.25, 0.50, 0.75, 1.0]
SEL_DELTA = 0.05 / 15                    # per-interval error: delta / (5 candidates x 3 groups)
SEL_GRID = np.linspace(0.0, 1.0, 401)    # grid for the time-uniform confidence sets


def clopper_pearson_total(k, n, total_alpha):
    """Clopper-Pearson interval with total noncoverage total_alpha, split equally."""
    if n == 0:
        return 0.0, 1.0
    from scipy.stats import beta as _beta
    a = total_alpha / 2.0
    lo = 0.0 if k == 0 else _beta.ppf(a, k, n - k + 1)
    hi = 1.0 if k == n else _beta.ppf(1 - a, k + 1, n - k)
    return float(lo), float(hi)


def time_uniform_mask(k, n, delta, grid=SEL_GRID):
    """Grid points p inside the anytime-valid (Jeffreys-mixture) confidence set after n draws."""
    from scipy.special import betaln
    if n == 0:
        return np.ones(len(grid), bool)
    out = np.full(grid.shape, betaln(0.5 + k, 0.5 + n - k) - betaln(0.5, 0.5), float)
    with np.errstate(divide="ignore", invalid="ignore"):
        if k > 0:
            out = out - k * np.where(grid > 0, np.log(np.where(grid > 0, grid, 1.0)), -np.inf)
        if n - k > 0:
            out = out - (n - k) * np.where(grid < 1, np.log(np.where(grid < 1, 1 - grid, 1.0)), -np.inf)
    return out < np.log(1.0 / delta)


def covgap_bounds(bounds, target):
    """Smallest and largest CovGap consistent with per-group coverage bounds (equations 13-14)."""
    L = [max(l - target, target - u, 0.0) for l, u in bounds]
    U = [max(abs(l - target), abs(u - target)) for l, u in bounds]
    return float(np.mean(L)), float(np.mean(U))


def certify(S, N, masks, target, time_uniform):
    """Index of the certified candidate, or None. Second value flags an inference failure."""
    bounds = []
    for c in range(len(SEL_W)):
        bc = []
        for j in range(G):
            if time_uniform:
                idx = np.flatnonzero(masks[c][j])
                if not idx.size:
                    return None, True
                bc.append((float(SEL_GRID[idx[0]]), float(SEL_GRID[idx[-1]])))
            else:
                bc.append(clopper_pearson_total(int(S[c, j]), int(N[c, j]), SEL_DELTA) if N[c, j] > 0 else (0., 1.))
        bounds.append(bc)
    L0 = covgap_bounds(bounds[0], target)[0]
    ok = [(covgap_bounds(bounds[c], target)[1], SEL_W[c], c) for c in range(1, len(SEL_W))
          if covgap_bounds(bounds[c], target)[1] < L0]
    return (sorted(ok)[0][2] if ok else None), False


def select_strength(cov_sel, g_sel, target):
    """Run every rule on the selection half. Returns {rule: (w, certified, units used, flag)}."""
    S, N = np.zeros((len(SEL_W), G)), np.zeros((len(SEL_W), G))
    masks = [[np.ones(len(SEL_GRID), bool) for _ in range(G)] for _ in SEL_W]
    early = None
    for t in range(len(g_sel)):
        j = g_sel[t]
        for c in range(len(SEL_W)):
            N[c, j] += 1
            S[c, j] += cov_sel[t, c]
            masks[c][j] = masks[c][j] & time_uniform_mask(int(S[c, j]), int(N[c, j]), SEL_DELTA)
        if early is None:
            p, fail = certify(S, N, masks, target, True)
            if fail:
                early = ("FAIL", t + 1)
            elif p is not None:
                early = (SEL_W[p], t + 1)
    n, out = len(g_sel), {}
    if (N[0] == 0).any():
        out["EMPIRICAL"] = (0.0, None, n, True)
    else:
        emp = [np.mean([abs(S[c, j] / N[c, j] - target) for j in range(G)]) for c in range(len(SEL_W))]
        out["EMPIRICAL"] = (SEL_W[int(np.argmin(np.round(emp, 12)))], None, n, False)
    p, fail = certify(S, N, masks, target, False)
    out["FIXED-CONFIDENCE"] = (0.0, False, n, fail) if (fail or p is None) else (SEL_W[p], True, n, False)
    p, fail = certify(S, N, masks, target, True)
    out["E-PROCESS-FULL"] = (0.0, False, n, fail) if (fail or p is None) else (SEL_W[p], True, n, False)
    if early is None:
        out["E-PROCESS-EARLY"] = (0.0, False, n, False)
    elif early[0] == "FAIL":
        out["E-PROCESS-EARLY"] = (0.0, False, early[1], True)
    else:
        out["E-PROCESS-EARLY"] = (early[0], True, early[1], False)
    return out


def selection_rows(base, choice, cov_t, len_t, g_t, target):
    """Evaluation-set results for each rule's choice and for every fixed candidate."""
    def score(c):
        per = [cov_t[g_t == j, c].mean() if (g_t == j).any() else np.nan for j in range(G)]
        return dict(coverage=cov_t[:, c].mean(), length=np.nanmean(len_t[:, c]),
                    covgap=np.nanmean(np.abs(np.array(per) - target)), fsc=np.nanmin(per),
                    **{"c%d" % j: per[j] for j in range(G)},
                    **{"l%d" % j: float(np.nanmean(len_t[g_t == j, c])) if (g_t == j).any() else np.nan
                       for j in range(G)})
    rows = [dict(base, selector=rule, w=w, certified=cert, sel_n=used, flag=flag, **score(SEL_W.index(w)))
            for rule, (w, cert, used, flag) in choice.items()]
    rows += [dict(base, selector="FIXED w=%.2f" % w, w=w, certified=None, sel_n=0, flag=False, **score(c))
             for c, w in enumerate(SEL_W)]
    return rows


def selection_candidates(res_cal, s_cal, res_sel, s_sel, res_t, s_t, alpha):
    """Calibrate each candidate on the calibration half; coverage on selection and evaluation units."""
    cov_sel = np.zeros((len(res_sel), len(SEL_W)), bool)
    cov_t, len_t = np.zeros((len(res_t), len(SEL_W)), bool), np.zeros((len(res_t), len(SEL_W)))
    for c, w in enumerate(SEL_W):
        lam = conformal_quantile(res_cal / s_cal(w), alpha)
        cov_sel[:, c] = res_sel <= lam * s_sel(w)
        h = lam * s_t(w)
        cov_t[:, c], len_t[:, c] = res_t <= h, 2 * h
    return cov_sel, cov_t, len_t


def selection_fits(X, y, a1, b1, seed):
    """Point predictor and mean-residual difficulty model, as in the main experiments."""
    fh = HGBR(**REG, random_state=seed).fit(X[a1], y[a1])
    res_b = np.abs(y[b1] - fh.predict(X[b1]))
    mm = HGBR(**REG, random_state=seed).fit(X[b1], res_b)
    floor = max(float(np.percentile(np.maximum(mm.predict(X[b1]), 0), 5)), 1e-6)
    norm = float(np.maximum(mm.predict(X[b1]), floor).mean())
    cuts = np.percentile(mm.predict(X[b1]), [33.3, 66.7])
    scale = lambda w, Xi: strength_scale(np.maximum(mm.predict(Xi), floor), w, norm)
    group = lambda Xi: np.clip(np.digitize(mm.predict(Xi), cuts), 0, G - 1)
    return fh, scale, group


def selection_clinical(D, cohort, s, alphas=ALPHAS):
    X, y, z, pid = D["X"], D["y"], D["z"], D["unit_id"]
    pats = np.unique(pid)
    pp = np.random.default_rng(3000 + s).permutation(pats)
    n = len(pp)
    dev, cal, te = set(pp[:int(.5 * n)]), pp[int(.5 * n):int(.8 * n)], set(pp[int(.8 * n):])
    order = np.random.default_rng(31000 + s).permutation(len(cal))          # before eligibility
    cc, sel = set(cal[order[:len(cal) // 2]]), set(cal[order[len(cal) // 2:]])
    M = {k: np.array([p in v for p in pid]) for k, v in [("dev", dev), ("cc", cc), ("sel", sel), ("te", te)]}
    di = np.flatnonzero(M["dev"])
    dp = np.unique(pid[di])
    np.random.default_rng(4000 + s).shuffle(dp)
    fitp = set(dp[:int(.6 * len(dp))])
    mf = np.array([p in fitp for p in pid[di]])
    a1, b1 = di[mf][z[di[mf]] == 1], di[~mf][z[di[~mf]] == 1]
    if len(a1) < 30 or len(b1) < 40:
        return []
    fh, scale, group = selection_fits(X, y, a1, b1, s)
    rs = np.random.default_rng(9500 + s)
    def one_per_patient(mask):
        ix = np.flatnonzero(mask)
        ix = ix[z[ix] == 1]
        return np.array([rs.choice(ix[pid[ix] == p]) for p in np.unique(pid[ix])]) if ix.size else np.array([], int)
    ci, si, ti = one_per_patient(M["cc"]), one_per_patient(M["sel"]), one_per_patient(M["te"])
    gen = {k: int(np.unique(pid[np.flatnonzero(M[k])]).size) for k in ["cc", "sel", "te"]}
    if len(ci) < 20 or len(si) < 10 or len(ti) < 10:
        return []
    res = lambda ix: np.abs(y[ix] - fh.predict(X[ix]))
    rows = []
    for alpha in alphas:
        cov_sel, cov_t, len_t = selection_candidates(
            res(ci), lambda w: scale(w, X[ci]), res(si), lambda w: scale(w, X[si]),
            res(ti), lambda w: scale(w, X[ti]), alpha)
        base = dict(dataset=cohort, alpha=alpha, split=s, n_cc=len(ci), n_sel=len(si), n_te=len(ti),
                    gen_cc=gen["cc"], gen_sel=gen["sel"], gen_te=gen["te"])
        rows += selection_rows(base, select_strength(cov_sel, group(X[si]), 1 - alpha),
                               cov_t, len_t, group(X[ti]), 1 - alpha)
    return rows


def selection_fre(D, s, alphas=ALPHAS):
    X, y, z = D["X"], D["y"], D["z"]
    n = len(y)
    pm = np.random.default_rng(s).permutation(n)
    dev, cal, te = pm[:int(.5 * n)], pm[int(.5 * n):int(.8 * n)], pm[int(.8 * n):]
    o = np.random.default_rng(31000 + s).permutation(len(cal))
    cc, sel = cal[o[:len(cal) // 2]], cal[o[len(cal) // 2:]]
    dd = dev.copy()
    np.random.default_rng(1000 + s).shuffle(dd)
    A, B = dd[:int(.6 * len(dd))], dd[int(.6 * len(dd)):]
    a1, b1 = A[z[A] == 1], B[z[B] == 1]
    ci, si, ti = cc[z[cc] == 1], sel[z[sel] == 1], te[z[te] == 1]
    if min(len(a1), len(b1), len(ci), len(si), len(ti)) < 30:
        return []
    fh, scale, group = selection_fits(X, y, a1, b1, s)
    res = lambda ix: np.abs(y[ix] - fh.predict(X[ix]))
    rows = []
    for alpha in alphas:
        cov_sel, cov_t, len_t = selection_candidates(
            res(ci), lambda w: scale(w, X[ci]), res(si), lambda w: scale(w, X[si]),
            res(ti), lambda w: scale(w, X[ti]), alpha)
        base = dict(dataset="freMTPL2", alpha=alpha, split=s, n_cc=len(ci), n_sel=len(si), n_te=len(ti),
                    gen_cc=len(cc), gen_sel=len(sel), gen_te=len(te))
        rows += selection_rows(base, select_strength(cov_sel, group(X[si]), 1 - alpha),
                               cov_t, len_t, group(X[ti]), 1 - alpha)
    return rows


def selection_synthetic(N, het, relev, signed, rep, alpha):
    """One synthetic replication. The calibration sample is halved; a fresh diagnostic draw of
    8,000 units measures population CovGap of each candidate (regret of the choice)."""
    D = synthetic.replication(N, het, relev, signed, rep)
    R, X, y, z, sig = D["roles"], D["X"], D["y"], D["z"], D["sigma"]
    Xd, yd, zd, sd = synthetic.draw(synthetic.world(rep, het, relev, signed), 8000, 850000 + rep)
    a1, b1 = R["A"][z[R["A"]] == 1], R["B"][z[R["B"]] == 1]
    c1, t1, d1 = R["C"][z[R["C"]] == 1], R["T"][z[R["T"]] == 1], np.flatnonzero(zd == 1)
    if min(len(a1), len(b1), len(t1)) < 30 or len(c1) < 20:
        return []
    o = np.random.default_rng(31000 + rep).permutation(len(c1))
    cc, sel = c1[o[:len(c1) // 2]], c1[o[len(c1) // 2:]]
    if len(cc) < 10 or len(sel) < 5:
        return []
    fh, scale, group = selection_fits(X, y, a1, b1, rep)
    target = 1 - alpha
    res = lambda ix: np.abs(y[ix] - fh.predict(X[ix]))
    cov_sel, cov_t, len_t = selection_candidates(
        res(cc), lambda w: scale(w, X[cc]), res(sel), lambda w: scale(w, X[sel]),
        res(t1), lambda w: scale(w, X[t1]), alpha)
    lam = [conformal_quantile(res(cc) / scale(w, X[cc]), alpha) for w in SEL_W]
    rd, gd = np.abs(yd[d1] - fh.predict(Xd[d1])), group(Xd[d1])
    dcov = np.array([[(rd <= lam[c] * scale(w, Xd[d1]))[gd == j].mean() if (gd == j).any() else np.nan
                      for j in range(G)] for c, w in enumerate(SEL_W)])
    pop = [float(np.nanmean(np.abs(dcov[c] - target))) for c in range(len(SEL_W))]
    g_true = np.clip(np.digitize(sig[t1], np.percentile(sd[d1], [33.3, 66.7])), 0, G - 1)
    nc = len(c1)
    base = dict(dataset="synthetic", alpha=alpha, N=N, het=het, relev=relev, signed=signed, rep=rep,
                split=rep, n_cc=len(cc), n_sel=len(sel), n_te=len(t1), gen_cc=nc // 2, gen_sel=nc - nc // 2,
                gen_te=len(t1), popD_const=pop[0], popD_best=float(np.nanmin(pop)), n_diag=len(d1))
    rows = selection_rows(base, select_strength(cov_sel, group(X[sel]), target), cov_t, len_t, group(X[t1]), target)
    for r in rows:
        c = SEL_W.index(r["w"])
        r["popD_sel"] = pop[c]
        r["regret"] = pop[c] - base["popD_best"]
        pt = [cov_t[g_true == j, c].mean() if (g_true == j).any() else np.nan for j in range(G)]
        r["covgap_true"] = float(np.nanmean(np.abs(np.array(pt) - target)))
    return rows


SEL_SYNTH_REPS = {0.20: 40, 0.10: 25, 0.05: 25}


def selection_experiment(which, splits=N_SPLITS, reps=None):
    if which == "clinical":
        data = {coh: load_ready(coh) for coh in ["PPMI", "XYZ"]}
        out = Parallel(n_jobs=N_JOBS)(delayed(selection_clinical)(data[c], c, s)
                                      for c in ["PPMI", "XYZ"] for s in range(splits))
    elif which == "fre":
        D = load_ready("freMTPL2")
        out = Parallel(n_jobs=N_JOBS)(delayed(selection_fre)(D, s) for s in range(splits))
    else:
        reps = reps or SEL_SYNTH_REPS
        tasks = [(N, h, r, sg, rep, a) for a in ALPHAS for N in [92, 361] for h in [0.0, 0.6]
                 for r in [0.8, 0.0, -0.8] for sg in [True, False] for rep in range(reps[a])]
        out = Parallel(n_jobs=N_JOBS)(delayed(selection_synthetic)(*t) for t in tasks)
    return pd.DataFrame([row for rows in out for row in rows])


selection_clin = run_or_load("selection_clinical", lambda: selection_experiment("clinical"), needs=["PPMI", "XYZ"])
selection_fre_ = run_or_load("selection_fre", lambda: selection_experiment("fre"), needs=["freMTPL2"])
selection_syn = run_or_load("selection_synthetic", lambda: selection_experiment("synthetic"))

## 9. Paper tables, figures and numbers

Everything the paper reports, rebuilt from the results above. Conventions:

* Metrics are computed within each split (replication for synthetic data) and averaged.
* Brackets and bands are 95% percentile bootstraps over splits: 4,000 resamples, seed 0.
* Table 6's "fraction shorter" pools all evaluated records. Table 8 drops empty intervals
  from the denominator.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

ROWS = [("Split / CASCADE-CRC (w=0)", "CASCADE-CRC CONSTANT"), ("Mondrian", "Mondrian (standard)"),
        ("CQR", "CQR"), ("CASCADE", "CASCADE (dev-beta)"), ("CASCADE-CRC (w=0.15)", "CASCADE-CRC SHRINK-0.15"),
        ("CASCADE-CRC (w=0.25)", "CASCADE-CRC SHRINK-0.25"), ("CASCADE-CRC (w=0.50)", "CASCADE-CRC SHRINK-0.50"),
        ("CASCADE-CRC (w=0.75)", "CASCADE-CRC SHRINK-0.75"), ("CASCADE-CRC (w=1)", "CASCADE-CRC STAGE2")]
STRENGTH_ARM = {0.0: "CASCADE-CRC CONSTANT", 1.0: "CASCADE-CRC STAGE2"}
arm_for = lambda w: STRENGTH_ARM.get(w, "CASCADE-CRC SHRINK-%.2f" % w)
COHORTS = [("PPMI", "PPMI"), ("XYZ Health", "XYZ")]
TARGETS = [(0.20, "80%"), (0.10, "90%"), (0.05, "95%")]
SCORE_ROWS = [("covariates", "Q-X"), ("raw probability", "Q-X+RAW-PROB"), ("VA probability", "Q-X+VA-PROB"),
              ("VA probability + width", "Q-X+VA-PROB+WIDTH"), ("VA upper", "Q-X+VA-P1"), ("VA endpoints", "Q-X+VA-P0P1")]


def bootstrap_ci(values, B=4000, seed=0):
    v = np.asarray(values, float)
    v = v[~np.isnan(v)]
    if len(v) < 2:
        return np.nan, np.nan
    draws = v[np.random.default_rng(seed).integers(0, len(v), (B, len(v)))].mean(1)
    return tuple(np.percentile(draws, [2.5, 97.5]))


def per_split(units, alpha, group_col="g_d2", n_groups=G):
    """Per-split coverage, length, CovGap, FSC, group coverages c0.. and group lengths l0.. ."""
    rows = []
    for s, x in units.groupby("split"):
        g = metrics.group_coverage(x.covered.values, x[group_col].values, n_groups)
        L = metrics.group_mean(x.set_length.values, x[group_col].values, n_groups)
        rows.append(dict(split=s, coverage=x.covered.mean(), length=x.set_length.mean(),
                         covgap=metrics.covgap(g, alpha), fsc=metrics.fsc(g),
                         **{"c%d" % j: g[j] for j in range(n_groups)}, **{"l%d" % j: L[j] for j in range(n_groups)}))
    return pd.DataFrame(rows).set_index("split")


def clinical_splits(units, cohort, alpha, arm, **kw):
    return per_split(units[(units.cohort == cohort) & (units.alpha == alpha) & (units.arm == arm)], alpha, **kw)


def fre_splits(df, alpha, arm):
    return df[(df.alpha == alpha) & (df.arm == arm)].set_index("split")


def synth_reps(df, alpha, arm):
    """Heterogeneous, PPMI-sized synthetic settings; CovGap and FSC on true-noise-scale groups."""
    x = df[(df.N == 361) & (df.het == 0.6) & (df.alpha == alpha) & (df.arm == arm)].set_index(["relev", "signed", "rep"])
    return x.rename(columns={"covgap_true": "covgap", "fsc_true": "fsc"})


TABLES = {}
TABLE_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(TABLE_DIR, exist_ok=True)


def show_table(title, df, digits):
    """Print a table and save it to results/tables/."""
    TABLES[title] = df
    slug = "".join(ch if ch.isalnum() else "_" for ch in title.lower()).strip("_")[:80]
    df.to_csv(os.path.join(TABLE_DIR, slug + ".csv"))
    print("\n" + title)
    print(df.to_string(float_format=lambda v: ("%." + str(digits) + "f") % v if np.isfinite(v) else "inf"))

### Tables 1 and 2: coverage, length, CovGap and FSC

In [ ]:
def summary_table(get_splits):
    out = {}
    for label, arm in ROWS:
        row = {}
        for alpha, t in TARGETS:
            P = get_splits(alpha, arm)
            for m in ["coverage", "length", "covgap", "fsc"]:
                row[(t, m)] = P[m].mean()
        out[label] = row
    return pd.DataFrame(out).T


if clinical_strength is not None:
    for name, coh in COHORTS:
        show_table("Table 1. %s" % name, summary_table(lambda a, arm: clinical_splits(clinical_strength, coh, a, arm)), 3)
if fre_strength is not None:
    show_table("Table 2. freMTPL2", summary_table(lambda a, arm: fre_splits(fre_strength, a, arm)), 3)
if synth_strength is not None:
    show_table("Table 2. Synthetic (heterogeneous, PPMI-sized; groups by true noise scale)",
               summary_table(lambda a, arm: synth_reps(synth_strength, a, arm)), 3)

### Tables 3 and 9: CovGap at 80% by difficulty score and strength

In [ ]:
W6 = [0.0, 0.15, 0.25, 0.50, 0.75, 1.0]


def score_table(primary, graded, baselines):
    """primary(w) and graded(signal, w) return a CovGap; baselines is {label: CovGap}."""
    T = pd.DataFrame({"mean absolute residual": {w: primary(w) for w in W6},
                      **{lab: {w: graded(sig, w) for w in W6} for lab, sig in SCORE_ROWS}}).T
    T.columns = ["w=%g" % w for w in W6]
    return T, baselines


def show_score_table(title, T, base):
    show_table(title, T, 4)
    TABLES[title + " baselines"] = pd.Series(base)
    pd.Series(base, name="covgap").to_csv(os.path.join(
        TABLE_DIR, "".join(ch if ch.isalnum() else "_" for ch in title.lower()).strip("_")[:80] + "_baselines.csv"))
    print("  baselines: " + ", ".join("%s %.4f" % kv for kv in base.items()))


if clinical_strength is not None and clinical_factorial is not None:
    for name, coh in COHORTS:
        cs = lambda arm: clinical_splits(clinical_strength, coh, 0.2, arm).covgap.mean()
        cf = lambda arm: clinical_splits(clinical_factorial, coh, 0.2, arm).covgap.mean()
        T, base = score_table(lambda w: cs(arm_for(w)), lambda sig, w: cf("SIG %s w=%.2f" % (sig, w)),
                              {"constant": cs("CASCADE-CRC CONSTANT"), "Mondrian": cs("Mondrian (standard)"), "CQR": cs("CQR")})
        show_score_table("Table 3. %s" % name, T, base)
if fre_strength is not None and fre_factorial is not None:
    fs = lambda arm: fre_splits(fre_strength, 0.2, arm).covgap.mean()
    ff = lambda arm: fre_splits(fre_factorial, 0.2, arm).covgap.mean()
    T, base = score_table(lambda w: fs(arm_for(w)), lambda sig, w: ff("SIG %s w=%.2f" % (sig, w)),
                          {"constant": ff("CASCADE-CRC CONSTANT"), "Mondrian": ff("Mondrian (standard)"), "CQR": ff("CQR")})
    show_score_table("Table 9. freMTPL2", T, base)
if synth_factorial is not None:
    sf = synth_factorial[synth_factorial.alpha == 0.2]
    sc = lambda arm: sf[sf.arm == arm].covgap_true.mean()
    T, base = score_table(lambda w: sc(arm_for(w)), lambda sig, w: sc("SIG %s w=%.2f" % (sig, w)),
                          {"constant": sc("CASCADE-CRC CONSTANT"), "Mondrian": sc("Mondrian (standard)"), "CQR": sc("CQR")})
    show_score_table("Table 9. Synthetic (%d replications)" % len(sf[sf.arm == "CQR"]), T, base)

### Table 4: paired differences from constant width

In [ ]:
def paired_table(get_splits):
    out = {}
    for label, arm in ROWS[4:]:
        row = {}
        for alpha, t in TARGETS:
            A, Z = get_splits(alpha, arm), get_splits(alpha, "CASCADE-CRC CONSTANT")
            d = A[["covgap", "fsc", "length"]] - Z.loc[A.index, ["covgap", "fsc", "length"]]
            lo, hi = bootstrap_ci(d.covgap.values)
            row.update({(t, "dCovGap"): d.covgap.mean(), (t, "lo"): lo, (t, "hi"): hi,
                        (t, "dFSC"): d.fsc.mean(), (t, "dLen"): d.length.mean()})
        out[label] = row
    return pd.DataFrame(out).T


if clinical_strength is not None:
    for name, coh in COHORTS:
        show_table("Table 4. %s" % name, paired_table(lambda a, arm: clinical_splits(clinical_strength, coh, a, arm)), 4)
if fre_strength is not None:
    show_table("Table 4. freMTPL2", paired_table(lambda a, arm: fre_splits(fre_strength, a, arm)), 4)
if synth_strength is not None:
    show_table("Table 4. Synthetic", paired_table(lambda a, arm: synth_reps(synth_strength, a, arm)), 4)

### Tables 5 and 6: coverage and length by difficulty stratum, and who gets shorter intervals

In [ ]:
def stratum_table(get_splits):
    out = {}
    for label, arm in ROWS:
        row = {}
        for alpha, t in TARGETS:
            P = get_splits(alpha, arm)
            for j, lab in enumerate(["low", "med", "high"]):
                row[(t, "cov " + lab)] = P["c%d" % j].mean()
            for j, lab in enumerate(["low", "med", "high"]):
                row[(t, "len " + lab)] = P["l%d" % j].mean()
        out[label] = row
    return pd.DataFrame(out).T


def fraction_shorter(units, cohort, alpha, arm):
    u = units[(units.cohort == cohort) & (units.alpha == alpha)]
    base = u[u.arm == "CASCADE-CRC CONSTANT"].set_index(["split", "patient"]).set_length
    x = u[u.arm == arm].set_index(["split", "patient"]).set_length
    return float((x < base.loc[x.index]).mean())


if clinical_strength is not None:
    for name, coh in COHORTS:
        show_table("Table 5. %s" % name, stratum_table(lambda a, arm: clinical_splits(clinical_strength, coh, a, arm)), 3)
        T6 = pd.DataFrame({label: {t: fraction_shorter(clinical_strength, coh, a, arm) for a, t in TARGETS}
                           for label, arm in ROWS[1:]}).T
        show_table("Table 6. %s: fraction of records shorter than constant width" % name, T6, 2)
if fre_strength is not None:
    show_table("Table 5. freMTPL2", stratum_table(lambda a, arm: fre_splits(fre_strength, a, arm)), 3)

### Table 7: PPMI grouped by previous dose (zero vs positive)

In [ ]:
if clinical_strength is not None:
    T7 = {}
    for label, arm in [r for r in ROWS if r[0] != "CASCADE-CRC (w=0.15)"]:
        row = {}
        for alpha, t in TARGETS:
            P = clinical_splits(clinical_strength, "PPMI", alpha, arm, group_col="g_pz", n_groups=2)
            row[(t, "CovGap")], row[(t, "FSC")] = P.covgap.mean(), P.fsc.mean()
        T7[label] = row
    show_table("Table 7. PPMI, previous-dose partition", pd.DataFrame(T7).T, 3)

### Table 8: how often intervals exclude zero (a decision to change the dose)

In [ ]:
if clinical_strength is not None:
    for name, coh in COHORTS:
        T8 = {}
        for label, arm in [("w=0", "CASCADE-CRC CONSTANT"), ("w=0.25", "CASCADE-CRC SHRINK-0.25"),
                           ("w=0.50", "CASCADE-CRC SHRINK-0.50"), ("w=1", "CASCADE-CRC STAGE2"),
                           ("CQR", "CQR"), ("Mondrian", "Mondrian (standard)")]:
            for alpha, t in TARGETS:
                u = clinical_strength[(clinical_strength.cohort == coh) & (clinical_strength.alpha == alpha) &
                                      (clinical_strength.arm == arm)]
                ne = ~u["empty"]
                ex = ((u.lo > 0) | (u.hi < 0))[ne]
                T8[(label, t)] = {"zero excluded": ex.groupby(u.split[ne]).mean().mean(),
                                  "empty": u["empty"].groupby(u.split).mean().mean(),
                                  **{lab: ex[u.g_d2[ne] == j].mean() for j, lab in enumerate(["low", "med", "high"])}}
        show_table("Table 8. %s" % name, pd.DataFrame(T8).T, 4)

### Table 10: units per split

In [ ]:
T10 = {}
if clinical_strength is not None:
    for name, coh in COHORTS:
        u = clinical_strength[(clinical_strength.cohort == coh) & (clinical_strength.alpha == 0.2) &
                              (clinical_strength.arm == "Split")]
        row = {"evaluation units": u.groupby("split").size().mean()}
        if os.path.exists(os.path.join(DATA_DIR, DATA_FILES[coh])):
            D = load_ready(coh)
            cal = [len(r["cal"]) for r in (clinical_split(D["unit_id"], D["z"], s) for s in range(N_SPLITS)) if r]
            row.update({"patients": len(np.unique(D["unit_id"])), "calibration units": np.mean(cal),
                        "nonzero rate": D["z"].mean()})
        T10[name] = row
if fre_strength is not None:
    f = fre_strength[(fre_strength.alpha == 0.2) & (fre_strength.arm == "Split")]
    T10["freMTPL2"] = {"calibration units": f.n_cal.mean(), "evaluation units": f.n_te.mean()}
if synth_strength is not None:
    for N in [361, 92]:
        x = synth_strength[(synth_strength.N == N) & (synth_strength.alpha == 0.2) & (synth_strength.arm == "Split")]
        T10["Synthetic N=%d" % N] = {"calibration units": x.n_eligible_cal.mean(), "evaluation units": x.n_eval.mean()}
if os.path.exists(os.path.join(DATA_DIR, DATA_FILES["freMTPL2"])):
    T10.setdefault("freMTPL2", {}).update({"patients": len(load_ready("freMTPL2")["y"]),
                                           "nonzero rate": load_ready("freMTPL2")["z"].mean()})
show_table("Table 10. Units per split (means; nonzero rate is the share of units with a recorded change)",
           pd.DataFrame(T10).T, 3)

### Figure 3: coverage and length by stratum as $w$ varies (80%)

Low- and high-difficulty curves cross where they receive equal coverage; beyond the
crossing (shaded) the ordering reverses.

In [ ]:
CURVE_W = [0.0, 0.05, 0.10, 0.15, 0.25, 0.50, 0.75, 1.0]


def stratum_curves():
    rows = []
    sources = []
    if clinical_strength is not None:
        sources += [(name, lambda w, coh=coh: clinical_splits(clinical_strength, coh, 0.2, arm_for(w))) for name, coh in COHORTS]
    if fre_strength is not None:
        sources += [("freMTPL2", lambda w: fre_splits(fre_strength, 0.2, arm_for(w)))]
    for name, get in sources:
        for w in CURVE_W:
            P = get(w)
            for j in range(G):
                lo, hi = bootstrap_ci(P["c%d" % j]); llo, lhi = bootstrap_ci(P["l%d" % j])
                rows.append(dict(data=name, w=w, stratum=j, cov=P["c%d" % j].mean(), lo=lo, hi=hi,
                                 len=P["l%d" % j].mean(), llo=llo, lhi=lhi))
    return pd.DataFrame(rows)


def crossing(d):
    lo, hi = d[d.stratum == 0].sort_values("w"), d[d.stratum == 2].sort_values("w")
    w, diff = lo.w.values, lo["cov"].values - hi["cov"].values
    for i in range(len(w) - 1):
        if diff[i] >= 0 and diff[i + 1] < 0:
            t = diff[i] / (diff[i] - diff[i + 1])
            return w[i] + t * (w[i + 1] - w[i])
    return None


def figure3(D, path):
    plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 8.6, "axes.linewidth": 0.8,
                         "legend.frameon": False})
    TEAL, ORANGE, MID = "#0E7C9B", "#D2691E", "#9AA3AA"
    names = [n for n in ["PPMI", "freMTPL2", "XYZ Health"] if n in set(D.data)]
    fig, A = plt.subplots(2, len(names), figsize=(2.4 * len(names), 4.4), sharex=True, squeeze=False,
                          gridspec_kw=dict(height_ratios=[1.25, 1]))
    units = {"PPMI": "log dose ratio", "freMTPL2": "log(1+claim)", "XYZ Health": "log dose ratio"}
    for k, name in enumerate(names):
        d = D[D.data == name]; wc = crossing(d); ax, bx = A[0][k], A[1][k]
        for a in (ax, bx):
            if wc is not None:
                a.axvspan(wc, 1.02, color="#F7EDEA", zorder=0, lw=0)
                a.axvline(wc, color="#8A6A64", ls="--", lw=0.9, zorder=1)
        if wc is not None:
            ax.text(wc + 0.03, 0.955, "over-\ncorrection", fontsize=6.6, color="#8A6A64", va="top", style="italic", linespacing=1.15)
            ax.text(wc + 0.02, 0.645, "$w\\approx%.2f$" % wc, fontsize=6.8, color="#5A4A46", ha="left", va="bottom")
        ax.axhline(0.80, color="#333333", ls=":", lw=1.0, zorder=1)
        for j, col, lab, ls, lw in [(0, TEAL, "lower difficulty", "-", 2.1), (1, MID, "medium", "--", 1.2),
                                    (2, ORANGE, "higher difficulty", "-", 2.1)]:
            s_ = d[d.stratum == j].sort_values("w")
            if j != 1:
                ax.fill_between(s_.w, s_.lo, s_.hi, color=col, alpha=0.16, lw=0, zorder=2)
                bx.fill_between(s_.w, s_.llo, s_.lhi, color=col, alpha=0.16, lw=0, zorder=2)
            ax.plot(s_.w, s_["cov"], color=col, ls=ls, lw=lw, marker=("o" if j != 1 else None), ms=3.2, zorder=3, label=lab)
            bx.plot(s_.w, s_["len"], color=col, ls=ls, lw=lw, marker=("o" if j != 1 else None), ms=3.2, zorder=3)
        ax.set_title(name, fontsize=9.2, fontweight="bold", pad=5); ax.set_xlim(-0.02, 1.02); ax.set_ylim(0.63, 0.97)
        bx.set_xlabel("adaptation strength $w$", fontsize=8.2)
        bx.text(0.02, 0.96, units[name], transform=bx.transAxes, fontsize=6.4, color="#777777", va="top", style="italic")
        for a in (ax, bx):
            a.set_xticks([0, 0.25, 0.5, 0.75, 1]); a.set_xticklabels(["0", ".25", ".5", ".75", "1"])
            a.grid(axis="y", ls="--", lw=0.4, alpha=0.35); a.set_axisbelow(True)
            for sp in ["top", "right"]:
                a.spines[sp].set_visible(False)
        if k > 0:
            ax.tick_params(labelleft=False)
    A[0][0].set_ylabel("group coverage", fontsize=8.4); A[1][0].set_ylabel("mean interval length", fontsize=8.4)
    A[0][0].text(1.0, 0.805, "80% target", fontsize=6.6, color="#333333", ha="right", va="bottom")
    h, l = A[0][0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=3, fontsize=7.8, bbox_to_anchor=(0.5, -0.035))
    fig.tight_layout(rect=[0, 0.045, 1, 1], h_pad=0.6)
    fig.savefig(path, bbox_inches="tight"); plt.close(fig)


curves = stratum_curves()
if len(curves):
    curves.to_csv(os.path.join(FIG_DIR, "figure3_data.csv"), index=False)
    figure3(curves, os.path.join(FIG_DIR, "figure3_crossing.pdf"))
    print("Figure 3 crossings:", {n: round(crossing(curves[curves.data == n]), 2) for n in curves.data.unique()})

### Figure 4: CovGap against strength at three targets

Stars mark the lowest observed CovGap among the evaluated strengths.

In [ ]:
def covgap_profiles():
    rows, sources = [], []
    if clinical_strength is not None:
        sources += [(name, lambda a, w, coh=coh: clinical_splits(clinical_strength, coh, a, arm_for(w))) for name, coh in COHORTS]
    if fre_strength is not None:
        sources += [("freMTPL2", lambda a, w: fre_splits(fre_strength, a, arm_for(w)))]
    for name, get in sources:
        for alpha, _ in TARGETS:
            for w in W6:
                P = get(alpha, w); lo, hi = bootstrap_ci(P.covgap)
                rows.append(dict(data=name, alpha=alpha, w=w, covgap=P.covgap.mean(), lo=lo, hi=hi))
    return pd.DataFrame(rows)


def figure4(D, path):
    plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 8.4, "axes.linewidth": 0.8,
                         "legend.frameon": False})
    COL = {0.20: "#0E7C9B", 0.10: "#7A6BA8", 0.05: "#D2691E"}
    names = [n for n in ["PPMI", "freMTPL2", "XYZ Health"] if n in set(D.data)]
    fig, axes = plt.subplots(1, len(names), figsize=(2.4 * len(names), 2.5), squeeze=False)
    for ax, name in zip(axes[0], names):
        for a in [0.20, 0.10, 0.05]:
            d = D[(D.data == name) & (D.alpha == a)].sort_values("w")
            ax.fill_between(d.w, d.lo, d.hi, color=COL[a], alpha=0.15, lw=0)
            ax.plot(d.w, d.covgap, color=COL[a], marker="o", ms=3.2, lw=1.8, label="%d%% coverage" % (100 * (1 - a)))
            k = int(np.argmin(d.covgap.values))
            ax.plot([d.w.values[k]], [d.covgap.values[k]], marker="*", ms=9, mfc="white", mec=COL[a], mew=1.4, zorder=5)
        ax.set_title(name, fontsize=9, fontweight="bold", pad=4); ax.set_xlabel("adaptation strength $w$", fontsize=8.2)
        ax.set_xticks([0, 0.25, 0.5, 0.75, 1]); ax.set_xticklabels(["0", ".25", ".5", ".75", "1"])
        ax.grid(axis="y", ls="--", lw=0.4, alpha=0.35); ax.set_axisbelow(True)
        for sp in ["top", "right"]:
            ax.spines[sp].set_visible(False)
    axes[0][0].set_ylabel("CovGap (within split, averaged)", fontsize=8.2)
    h, l = axes[0][0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=3, fontsize=7.8, bbox_to_anchor=(0.5, -0.06))
    fig.tight_layout(rect=[0, 0.08, 1, 1]); fig.savefig(path, bbox_inches="tight"); plt.close(fig)


profiles = covgap_profiles()
if len(profiles):
    profiles.to_csv(os.path.join(FIG_DIR, "figure4_data.csv"), index=False)
    figure4(profiles, os.path.join(FIG_DIR, "figure4_covgap.pdf"))
    print("Figures written to", FIG_DIR)

### Appendix C and the Discussion: strength selection

Mean evaluation CovGap of each rule's choice against constant width, and how often the
certifying rules certified.

In [ ]:
def selection_summary(df, key, label, covcol="covgap"):
    rows = []
    for alpha, t in TARGETS:
        x = df[df.alpha == alpha]
        e = x[x.selector == "EMPIRICAL"].set_index(key)[covcol]
        z = x[x.selector == "FIXED w=0.00"].set_index(key)[covcol]
        cert = {r: int(x[x.selector == r].certified.astype(str).eq("True").sum())
                for r in ["FIXED-CONFIDENCE", "E-PROCESS-FULL", "E-PROCESS-EARLY"]}
        rows.append(dict(dataset=label, target=t, empirical=e.mean(), constant=z.loc[e.index].mean(),
                         improved=e.mean() < z.loc[e.index].mean(), runs=len(e), **{"certified " + k: v for k, v in cert.items()}))
    return rows


S = []
if selection_clin is not None:
    for name, coh in COHORTS:
        S += selection_summary(selection_clin[selection_clin.dataset == coh], ["split"], name)
if selection_fre_ is not None:
    S += selection_summary(selection_fre_, ["split"], "freMTPL2")
if selection_syn is not None:
    S += selection_summary(selection_syn, ["N", "het", "relev", "signed", "rep"], "Synthetic")
if S:
    S = pd.DataFrame(S)
    print(S.to_string(index=False, float_format=lambda v: "%.4f" % v))
    at80 = S[S.target == "80%"]
    print("\nAt 80%%: the empirical rule improved coverage balance on %d of %d datasets; "
          "certifying rules certified in %d of %d runs."
          % (at80.improved.sum(), len(at80), int(S.filter(like="certified").values.sum()), int(3 * S.runs.sum())))

### Numbers quoted in the text, captions and appendices

In [ ]:
def numbers_in_text():
    N = {}
    if clinical_strength is not None:
        P = lambda coh, a, arm: clinical_splits(clinical_strength, coh, a, arm)
        c0, c5 = P("PPMI", .2, "CASCADE-CRC CONSTANT"), P("PPMI", .2, "CASCADE-CRC SHRINK-0.50")
        N["PPMI 80%: CovGap w=0 -> w=0.50 reduction (%)"] = 100 * (1 - c5.covgap.mean() / c0.covgap.mean())
        dev = lambda p: np.mean(np.abs(p[["c0", "c1", "c2"]].mean().values - 0.8))
        N["PPMI 80%: deviation of split-averaged group coverage, reduction (%)"] = 100 * (1 - dev(c5) / dev(c0))
        x = P("XYZ", .1, "CASCADE-CRC CONSTANT")
        N["XYZ 90% constant: split-averaged group coverage"] = tuple(np.round(x[["c0", "c1", "c2"]].mean().values, 3))
        N["XYZ 90% constant: their mean deviation"] = float(np.mean(np.abs(x[["c0", "c1", "c2"]].mean().values - 0.9)))
        g25 = P("PPMI", .2, "CASCADE-CRC SHRINK-0.25")[["c0", "c2"]].mean()
        N["PPMI 80% w=0.25: low and high coverage, gap"] = (round(g25.c0, 3), round(g25.c2, 3), round(g25.c0 - g25.c2, 3))
        for name, coh in COHORTS:
            u = clinical_strength[(clinical_strength.cohort == coh) & (clinical_strength.alpha == .2) &
                                  (clinical_strength.arm == "Split")]
            N["%s evaluated units per split by stratum" % name] = tuple(np.round(u.groupby(["split", "g_d2"]).size().unstack().mean().values, 1))
            for a in [0.1, 0.05]:
                m = clinical_strength[(clinical_strength.cohort == coh) & (clinical_strength.alpha == a) &
                                      (clinical_strength.arm == "Mondrian (standard)")]
                N["%s Mondrian infinite intervals at %.0f%% (%%)" % (name, 100 * (1 - a))] = 100 * m.infinite.mean()
        u = clinical_strength[(clinical_strength.cohort == "PPMI") & (clinical_strength.alpha == .2) & (clinical_strength.arm == "Split")]
        N["PPMI units per split with previous dose positive / zero"] = (
            round(u[u.g_pz == 0].groupby("split").size().mean(), 1), round(u[u.g_pz == 1].groupby("split").size().mean(), 1))
        # Appendix F: dose-scale interval widths, PPMI 80% (lower end floored at zero)
        pp = clinical_strength[(clinical_strength.cohort == "PPMI") & (clinical_strength.alpha == .2)]
        def dose_range(arm):
            x = pp[pp.arm == arm]
            return x, (x.prev + 1) * np.exp(x.hi) - 1 - np.maximum((x.prev + 1) * np.exp(x.lo) - 1, 0.0)
        x0, r0 = dose_range("CASCADE-CRC CONSTANT"); x5, r5 = dose_range("CASCADE-CRC SHRINK-0.50")
        bins = pd.cut(x0.prev, [0, 1, 300, 600, np.inf], right=False)
        N["Appendix F: w=0 median dose range by previous dose 0, [1,300), [300,600), 600+ (mg/day)"] = tuple(
            np.round(r0.groupby(bins, observed=True).median().values, 0))
        N["Appendix F: median dose range high stratum w=0 -> w=0.50"] = (round(r0[x0.g_d2 == 2].median()), round(r5[x5.g_d2 == 2].median()))
        N["Appendix F: median dose range low stratum w=0 -> w=0.50"] = (round(r0[x0.g_d2 == 0].median()), round(r5[x5.g_d2 == 0].median()))
        N["Appendix F: overall median dose range w=0 -> w=0.50"] = (round(r0.median()), round(r5.median()))
        # Appendix A.2: size-stratified coverage violation
        def sscv(x, alpha):
            L, cov = x.set_length.values, x.covered.values
            if np.ptp(L) <= 1e-9 * abs(np.median(L)):
                return abs(cov.mean() - (1 - alpha))
            b = np.digitize(L, np.unique(np.quantile(L, [0.25, 0.5, 0.75])))
            return max(abs(cov[b == k].mean() - (1 - alpha)) for k in np.unique(b) if (b == k).sum() >= 20)
        for arm in ["CASCADE-CRC CONSTANT", "Mondrian (standard)", "CASCADE-CRC SHRINK-0.50", "CQR"]:
            x = pp[pp.arm == arm]
            N["Appendix A.2: SSCV PPMI 80%%, %s" % arm] = x.groupby("split").apply(lambda s: sscv(s, .2)).mean()
    if clinical_factorial is not None and fre_factorial is not None and synth_factorial is not None:
        spread = []
        for a, _ in TARGETS:
            for w in W6:
                for coh in ["PPMI", "XYZ"]:
                    v = [clinical_splits(clinical_factorial, coh, a, "SIG %s w=%.2f" % (sg, w)).covgap.mean() for _, sg in SCORE_ROWS]
                    spread.append((a, max(v) - min(v)))
                f = fre_factorial[fre_factorial.alpha == a]
                v = [f[f.arm == "SIG %s w=%.2f" % (sg, w)].covgap.mean() for _, sg in SCORE_ROWS]; spread.append((a, max(v) - min(v)))
                f = synth_factorial[synth_factorial.alpha == a]
                v = [f[f.arm == "SIG %s w=%.2f" % (sg, w)].covgap_true.mean() for _, sg in SCORE_ROWS]; spread.append((a, max(v) - min(v)))
        N["Largest CovGap spread among the six quantile scores (any dataset, target, strength)"] = max(v for _, v in spread)
        N["... at 90% and 95% only"] = max(v for a, v in spread if a != 0.2)
        sf = synth_factorial[synth_factorial.alpha == 0.2]
        best_mean = min(sf[sf.arm == arm_for(w)].covgap_true.mean() for w in W6)
        best_q = min(sf[sf.arm == "SIG %s w=%.2f" % (sg, w)].covgap_true.mean() for _, sg in SCORE_ROWS for w in W6)
        N["Synthetic: best mean-residual vs best quantile configuration, difference"] = abs(best_mean - best_q)
    if synth_strength is not None:
        x = synth_strength[(synth_strength.N == 361) & (synth_strength.het == 0.6) & (synth_strength.alpha == .2) & (synth_strength.arm == "Split")]
        N["Synthetic Table 2 replications"] = len(x)
        N["Synthetic eligible calibration units (N=361) / evaluation units"] = (round(x.n_eligible_cal.mean(), 1), round(x.n_eval.mean(), 1))
    return N


def _fmt(v):
    if isinstance(v, tuple):
        return "(" + ", ".join("%g" % float(x) for x in v) + ")"
    return ("%.4f" % v) if isinstance(v, float) else str(v)


for k, v in numbers_in_text().items():
    print("%-92s %s" % (k, _fmt(v)))

## 10. Reproduction check

Reruns a few repetitions with the code above and compares them with the stored results:
interval endpoints for the clinical data (unit by unit), and coverage, length and group
coverage for freMTPL2 and the synthetic data. Stored labels are mapped with `legacy_label`
(section 1). The synthetic check turns `ORACLE_AS_SUBMITTED` on, so it compares like with
like.

In [ ]:
def compare(new, old, key, cols, tol=1e-9):
    """Largest absolute difference per column after matching rows on `key`."""
    if "arm" in old:
        old = old.assign(arm=old.arm.map(legacy_label))
    m = new.merge(old, on=key, suffixes=("", "_old"))
    report = {"stored rows": len(old), "matched": len(m)}
    for col in cols:
        a, b = m[col].astype(float).values, m[col + "_old"].astype(float).values
        same_inf = (np.isinf(a) == np.isinf(b)) & (np.isnan(a) == np.isnan(b))
        fin = np.isfinite(a) & np.isfinite(b)
        report[col] = float(np.max(np.abs(a[fin] - b[fin]))) if fin.any() else 0.0
        report[col + " inf/nan mismatches"] = int((~same_inf).sum())
    report["ok"] = len(m) == len(old) and all(
        (report[col] <= tol) and report[col + " inf/nan mismatches"] == 0 for col in cols)
    return report


def show(name, legacy, r, cols):
    worst = max(r[c] for c in cols)
    bad_inf = sum(r[c + " inf/nan mismatches"] for c in cols)
    print("%-19s vs %-20s %7d of %7d stored rows matched | max |diff| %.1e | inf/nan mismatches %d | %s"
          % (name, legacy, r["matched"], r["stored rows"], worst, bad_inf, "OK" if r["ok"] else "MISMATCH"))


SYN_KEY = ["N", "het", "relev", "signed", "rep", "alpha", "arm"]
SYN_COLS = ["coverage", "length", "covgap_true", "fsc_true", "covgap_pred", "fsc_pred", "ct0", "ct1", "ct2"]


def check_synthetic(n_reps=3):
    global ORACLE_AS_SUBMITTED
    saved, ORACLE_AS_SUBMITTED = ORACLE_AS_SUBMITTED, True     # compare like with like
    for name, legacy, oracles, settings in [
            ("synth_strength", "synth_v2", ["ORACLE-noise", "ORACLE-residual-q"], SYNTH_SETTINGS),
            ("synth_factorial", "synth_factorial", ["ORACLE-residual-q"],
             dict(SYNTH_SETTINGS, budgets=[("PPMI-sized-sensitivity", 361)], hets=[("heterogeneous", 0.6)]))]:
        new = synthetic_experiment(name, oracles, settings, reps=n_reps)
        old = pd.read_csv(os.path.join(LEGACY_DIR, legacy + ".csv.gz"))
        old = old[old.rep < n_reps]
        show(name, legacy, compare(new, old, SYN_KEY, SYN_COLS), SYN_COLS)
    ORACLE_AS_SUBMITTED = saved


def check_clinical(n_splits=2):
    pairs = [("clinical_strength", ["fine_clinical", "lowW_clinical", "alpha_ext_clinical"]),
             ("clinical_factorial", ["factorial_clinical"])]
    for name, legacies in pairs:
        new = clinical_experiment(name, splits=n_splits)
        for legacy in legacies:
            old = pd.read_csv(os.path.join(LEGACY_DIR, legacy + ".csv.gz"), low_memory=False)
            old = old[old.split < n_splits].replace({"cohort": {"UF": "XYZ"}})
            show(name, legacy, compare(new, old, ["cohort", "alpha", "split", "arm", "patient"],
                                       ["lo", "hi"]), ["lo", "hi"])


def check_fre(n_splits=2):
    for name, legacies in [("fre_strength", ["fre_lowW", "fre_all_alpha"]), ("fre_factorial", ["factorial_fre"])]:
        new = fre_experiment(name, splits=n_splits)
        for legacy in legacies:
            old = pd.read_csv(os.path.join(LEGACY_DIR, legacy + ".csv.gz"))
            old = old[old.split < n_splits]
            cols = ["coverage", "length", "covgap", "c0", "c1", "c2"]
            show(name, legacy, compare(new, old, ["alpha", "split", "arm"], cols), cols)

def check_selection(n_splits=3, n_reps=3):
    """Strength selection against the stored selector results (clinical and synthetic)."""
    cols = ["w", "sel_n", "coverage", "length", "covgap", "fsc"]
    if all(os.path.exists(os.path.join(DATA_DIR, DATA_FILES[c])) for c in ["PPMI", "XYZ"]):
        new = selection_experiment("clinical", splits=n_splits)
        old = pd.read_csv(os.path.join(LEGACY_DIR, "selector_results.csv.gz"))
        old = old[old.split < n_splits].replace({"dataset": {"UF": "XYZ"}})
        show("selection_clinical", "selector_results",
             compare(new, old, ["dataset", "alpha", "split", "selector"], cols), cols)
    reps = {a: n_reps for a in ALPHAS}
    new = selection_experiment("synthetic", reps=reps)
    old = pd.read_csv(os.path.join(LEGACY_DIR, "selector_synthetic.csv.gz"))
    old = old[old.rep < n_reps]
    cols = cols + ["popD_sel", "regret", "covgap_true"]
    show("selection_synthetic", "selector_synthetic",
         compare(new, old, ["N", "het", "relev", "signed", "rep", "alpha", "selector"], cols), cols)

In [ ]:
RUN_CHECKS = os.environ.get("CASCADE_RUN_CHECKS", "synthetic")   # e.g. "synthetic,clinical,fre"
if "synthetic" in RUN_CHECKS:
    check_synthetic()
if "clinical" in RUN_CHECKS:
    check_clinical()
if "fre" in RUN_CHECKS:
    check_fre()
if "selection" in RUN_CHECKS:
    check_selection()